# Midjourney via TTAPI – Google Colab

Generate Midjourney images using **TTAPI** (same API as the junk journal app).

**Flow:**
1. Install dependencies and set **TTAPI API key** and **domain**.
2. **Single prompt**: submit one `/imagine`, poll until complete, display image(s).
3. **Bulk prompts**: submit multiple prompts, poll each, display all; optionally **5a. Regenerate failed** (banned words, 400 errors) via ChatGPT.
4. **Filter by position** (1st / 2nd / 3rd / 4th / All) – same as the app’s collection view.
5. **Download ZIP** or **Upload to Google Drive** using the filtered list (shuffled).
6. **Grid Generator** – create 3×4 grid layouts (12 images per page) from filtered images.

## 1. Install dependencies

In [ ]:
!pip install -q requests Pillow

## 2. Set TTAPI API key and domain

Get your API key from [ttapi.io](https://ttapi.io). Default: **Hold Account** (https://hold.ttapi.io). For PPU use https://api.ttapi.io.

In [ ]:
TTAPI_API_KEY = ""  # Paste your key from https://ttapi.io
TTAPI_DOMAIN = "https://hold.ttapi.io"  # Hold Account (or "https://api.ttapi.io" for PPU)

MODE = "relax"  # "fast" | "relax" | "turbo" (same as app) – default relax

## 3. TTAPI helpers (imagine, fetch, poll)

Uses the same API as the app: `getUImages: true`, `mode`, and standard fetch response parsing.

In [ ]:
import time
import requests


def get_ttapi_account_count(api_key, domain="https://hold.ttapi.io", mode="relax"):
    """Get the number of available TTAPI accounts for dynamic batch sizing."""
    try:
        domain = domain.rstrip("/")
        url = f"{domain}/midjourney/v1/accounts"
        r = requests.get(
            url,
            headers={"TT-API-KEY": api_key, "Content-Type": "application/json"},
            timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        accounts = data.get("accounts") or data.get("data") or []
        if not isinstance(accounts, list):
            return 1
        
        # If fast mode, only count accounts with Fast Time available
        if mode.lower() == "fast":
            accounts_with_fast = [acc for acc in accounts if acc.get("fast_time_remaining", 0) > 0 or acc.get("has_fast_time", False)]
            return max(len(accounts_with_fast) if accounts_with_fast else len(accounts), 1)
        
        return max(len(accounts), 1)
    except Exception as e:
        print(f"Could not fetch TTAPI accounts: {e}. Defaulting to 1 account.")
        return 1


def ttapi_imagine(prompt, api_key, domain="https://hold.ttapi.io", mode="relax"):
    """Submit one /imagine job. Same payload as app: prompt, getUImages, mode. Always sends mode so relax/turbo are applied."""
    domain = domain.rstrip("/")
    mode = (mode or "relax").lower()
    # HOLD accounts (hold.ttapi.io) need --relax in the prompt for relax mode (per TTAPI docs)
    if "hold.ttapi.io" in domain and mode == "relax" and "--relax" not in prompt.lower():
        prompt = prompt.strip() + " --relax"
    body = {"prompt": prompt, "getUImages": True, "mode": mode}
    url = f"{domain}/midjourney/v1/imagine"
    r = requests.post(
        url,
        headers={"TT-API-KEY": api_key, "Content-Type": "application/json"},
        json=body,
        timeout=90,
    )
    r.raise_for_status()
    return r.json()


def ttapi_fetch(job_id, api_key, domain="https://hold.ttapi.io", timeout=60):
    """Poll job status. Longer timeout for relax mode (Hold can be slow)."""
    url = f"{domain.rstrip('/')}/midjourney/v1/fetch"
    r = requests.get(
        url,
        params={"jobId": job_id},
        headers={"TT-API-KEY": api_key, "Content-Type": "application/json"},
        timeout=timeout,
    )
    r.raise_for_status()
    return r.json()


def extract_image_urls(status):
    """Extract image URL(s) from TTAPI fetch response (same logic as app)."""
    urls = []
    data = status.get("data") or {}
    if isinstance(data, dict):
        if isinstance(data.get("images"), list) and data["images"]:
            urls = data["images"]
        elif isinstance(data.get("image_urls"), list) and data["image_urls"]:
            urls = data["image_urls"]
        elif isinstance(data.get("urls"), list) and data["urls"]:
            urls = data["urls"]
        elif data.get("cdnImage"):
            urls = [data["cdnImage"]]
        elif data.get("discordImage"):
            urls = [data["discordImage"]]
        elif data.get("image_url"):
            urls = [data["image_url"]]
        elif data.get("image"):
            urls = [data["image"]]
        elif data.get("url"):
            urls = [data["url"]]
    for u in (status.get("images") or []), (status.get("urls") or []):
        if isinstance(u, list) and u and not urls:
            urls = u
            break
    if not urls and status.get("image"):
        urls = [status["image"]]
    if not urls and status.get("url"):
        urls = [status["url"]]
    return [u for u in urls if isinstance(u, str) and u.startswith("http")]


def poll_until_done(job_id, api_key, domain, poll_interval=5.0, max_wait=600.0, fetch_timeout=60, fetch_retries=3):
    """Poll until job completes or fails. Relax mode can be slow: longer max_wait and fetch timeout, retry on read timeout."""
    start = time.time()
    while time.time() - start < max_wait:
        for attempt in range(fetch_retries):
            try:
                data = ttapi_fetch(job_id, api_key, domain, timeout=fetch_timeout)
                break
            except requests.exceptions.Timeout:
                if attempt < fetch_retries - 1:
                    time.sleep(5)
                    continue
                raise
        # TTAPI returns status at top level and/or inside data
        status = (data.get("status") or (data.get("data") or {}).get("status") or data.get("state") or "").lower()
        # TTAPI uses "SUCCESS" when done (not "complete")
        if "complete" in status or "succeed" in status or status == "success":
            return data, extract_image_urls(data)
        if "fail" in status or "error" in status:
            return data, []
        time.sleep(poll_interval)
    return {"status": "timeout", "jobId": job_id}, []

## 4. Single prompt – submit and display

In [ ]:
from IPython.display import display, HTML
from html import escape

def display_images_4_per_row(image_urls):
    """Display images in a grid: 4 per row, then next row of 4."""
    for start in range(0, len(image_urls), 4):
        chunk = image_urls[start : start + 4]
        imgs = "".join(f'<img src="{escape(url)}" style="width: 100%; display: block;" />' for url in chunk)
        display(HTML(f'<div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 16px;">{imgs}</div>'))

SINGLE_PROMPT = "a cozy junk journal page, vintage paper, dried flowers --ar 3:4 --v 6.1"

if not TTAPI_API_KEY:
    print("Set TTAPI_API_KEY in the config cell above.")
else:
    resp = ttapi_imagine(SINGLE_PROMPT, TTAPI_API_KEY, TTAPI_DOMAIN, MODE)
    job_id = (resp.get("data") or {}).get("jobId") or resp.get("jobId") or resp.get("id")
    if not job_id:
        print("No jobId in response:", resp)
    else:
        print(f"Job submitted: {job_id}. Polling...")
        status, image_urls = poll_until_done(job_id, TTAPI_API_KEY, TTAPI_DOMAIN)
        if image_urls:
            display_images_4_per_row(image_urls)
            single_result = {"status": status.get("status"), "image_urls": image_urls}
        else:
            print("No images:", status)

## 5. Bulk prompts – paste list and run

Paste one prompt per line (or use a list). Each job is submitted then polled in order.

**Why some prompts show 0 images or an error:** (1) **Banned words** – TTAPI/Midjourney rejects the prompt (400); (2) **Job failed** – job was accepted but failed on the server (content policy, etc.); (3) **Timeout** – job didn’t finish within ~10 min. Use **Section 5a** to rephrase failed prompts via ChatGPT and re-submit.

In [ ]:
# One prompt per line; can include image URL + text (e.g. from WordPress step in Arcane Splitter)
BULK_PROMPTS = """
a vintage botanical illustration, herbs and flowers --ar 3:4 --v 6.1
old paper with ink sketches, coffee stains --ar 3:4 --v 6.1
""".strip().split("\n")
BULK_PROMPTS = [p.strip() for p in BULK_PROMPTS if p.strip()]

# Or set explicitly:
# BULK_PROMPTS = [
#     "https://example.com/ref.png style reference prompt --ar 3:4 --v 6.1",
# ]

In [ ]:
import time
import random
from IPython.display import display, HTML
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed

def display_images_4_per_row(image_urls):
    """Display images in a grid: 4 per row, then next row of 4."""
    for start in range(0, len(image_urls), 4):
        chunk = image_urls[start : start + 4]
        imgs = "".join(f'<img src="{escape(url)}" style="width: 100%; display: block;" />' for url in chunk)
        display(HTML(f'<div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 16px;">{imgs}</div>'))

RATE_LIMIT_RETRIES = 5   # Retry imagine up to 5 times on rate limit
BASE_WAIT_TIME = 60   # Base wait time in seconds (will be multiplied exponentially)
MAX_WAIT_TIME = 300   # Maximum wait time (5 minutes)

def _is_rate_limit_error(e):
    """True if HTTPError is due to rate limiting (429 or body says 'rate limit')."""
    if e.response is None:
        error_str = str(e).lower()
        return "rate limit" in error_str or "rate limited" in error_str or "429" in error_str
    if e.response.status_code == 429:
        return True
    body = (e.response.text or "").lower()
    try:
        j = e.response.json()
        msg = str(j.get("message", j.get("error", ""))).lower()
        body += msg
    except Exception:
        pass
    error_str = str(e).lower()
    return ("rate limit" in body or "rate limited" in body or 
            "rate limit" in error_str or "rate limited" in error_str or
            "429" in body or "429" in error_str)

def _exponential_backoff_wait(attempt, base_wait=BASE_WAIT_TIME, max_wait=MAX_WAIT_TIME):
    """Calculate exponential backoff wait time with jitter."""
    wait = min(base_wait * (2 ** attempt), max_wait)
    # Add jitter: random 10-20% of wait time
    jitter = wait * random.uniform(0.1, 0.2)
    return wait + jitter

def process_one_prompt(args):
    """Submit one prompt, poll until done. Returns (index, result_dict). Retries on rate limit with exponential backoff."""
    i, prompt = args
    try:
        resp = None
        last_rate_limit_error = None
        for rate_attempt in range(RATE_LIMIT_RETRIES + 1):
            try:
                resp = ttapi_imagine(prompt, TTAPI_API_KEY, TTAPI_DOMAIN, MODE)
                break
            except requests.exceptions.HTTPError as e:
                if _is_rate_limit_error(e) and rate_attempt < RATE_LIMIT_RETRIES:
                    last_rate_limit_error = str(e)
                    wait_time = _exponential_backoff_wait(rate_attempt)
                    print(f"    Prompt {i + 1}: Rate limited (attempt {rate_attempt + 1}/{RATE_LIMIT_RETRIES + 1}), waiting {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    continue
                raise
        if resp is None:
            error_msg = last_rate_limit_error or "No response after retries"
            return (i, {"prompt": prompt[:80], "error": error_msg, "image_urls": []})
        job_id = (resp.get("data") or {}).get("jobId") or resp.get("jobId") or resp.get("id")
        if not job_id:
            return (i, {"prompt": prompt[:80], "error": "No jobId", "image_urls": []})
        status, image_urls = poll_until_done(job_id, TTAPI_API_KEY, TTAPI_DOMAIN)
        st = status.get("status") or (status.get("data") or {}).get("status")
        # If we got 0 images, treat as failure so 5a can retry/rephrase (job failed, timed out, or empty response)
        err = None
        if not image_urls:
            st_str = (st or "").lower() if isinstance(st, str) else str(st)
            msg = (status.get("message") or (status.get("data") or {}).get("message") or "")
            if "timeout" in st_str or status.get("status") == "timeout":
                err = "Job timed out (no images)"
            elif "fail" in st_str or "error" in st_str:
                err = msg or f"Job failed: {st}"
            else:
                err = "No images returned (job may have failed or timed out)"
        return (i, {
            "prompt": prompt[:80], "jobId": job_id,
            "status": st, "image_urls": image_urls,
            "imageUrl": image_urls[0] if image_urls else None,
            **({"error": err} if err else {}),
        })
    except requests.exceptions.HTTPError as e:
        msg = str(e)
        if e.response is not None and e.response.status_code == 400:
            try:
                body = e.response.json()
                msg = body.get("error", body.get("message", msg))
            except Exception:
                msg = e.response.text[:200] if e.response.text else msg
            msg = f"400 Bad Request: {msg} (check prompt for banned words or invalid params)"
        elif _is_rate_limit_error(e):
            msg = "The resource is being rate limited."
        return (i, {"prompt": prompt[:80], "error": msg, "image_urls": []})
    except Exception as e:
        return (i, {"prompt": prompt[:80], "error": str(e), "image_urls": []})

# Adaptive concurrency: start conservative, reduce if rate limits occur
MAX_CONCURRENT = 2  # Use 3 if rate limit is rare; use 1 for strict limits (slower but fewer failures)
BATCH_DELAY = 5  # Increased delay between batches (seconds)

if not TTAPI_API_KEY:
    print("Set TTAPI_API_KEY in the config cell above.")
elif not BULK_PROMPTS:
    print("Add prompts in BULK_PROMPTS in the previous cell.")
else:
    n = len(BULK_PROMPTS)
    total_batches = (n + MAX_CONCURRENT - 1) // MAX_CONCURRENT
    ttapi_results = [None] * n
    rate_limit_count = 0
    print(f"Processing {n} prompt(s) in {total_batches} batch(es) with {MAX_CONCURRENT} concurrent requests per batch...")
    print(f"Rate limit handling: {RATE_LIMIT_RETRIES} retries with exponential backoff (base: {BASE_WAIT_TIME}s, max: {MAX_WAIT_TIME}s)")

    for batch_start in range(0, n, MAX_CONCURRENT):
        batch_end = min(batch_start + MAX_CONCURRENT, n)
        batch_num = batch_start // MAX_CONCURRENT + 1
        batch_size = batch_end - batch_start
        print(f"[Ttapi Bulk] Starting batch {batch_num}/{total_batches}: {batch_size} prompt(s) in parallel (prompts {batch_start + 1}-{batch_end} of {n})...")

        batch_args = [(i, BULK_PROMPTS[i]) for i in range(batch_start, batch_end)]
        with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as executor:
            futures = {executor.submit(process_one_prompt, a): a[0] for a in batch_args}
            for future in as_completed(futures):
                idx, result = future.result()
                ttapi_results[idx] = result
                urls = result.get("image_urls") or []
                err = result.get("error")
                if err:
                    if "rate limit" in str(err).lower():
                        rate_limit_count += 1
                    print(f"  Prompt {idx + 1}: Error - {err}")
                else:
                    print(f"  Prompt {idx + 1}: {len(urls)} image(s)")

        print(f"[Ttapi Bulk] Batch {batch_num}/{total_batches} completed.")
        if batch_end < n:
            # Adaptive delay: longer if we're seeing rate limits
            delay = BATCH_DELAY * (2 if rate_limit_count > batch_num * 0.3 else 1)
            time.sleep(delay)

    # Second pass: retry rate-limited prompts one at a time with longer delays
    rate_limited = [i for i in range(n) if ttapi_results[i] and ttapi_results[i].get("error") and "rate limit" in str(ttapi_results[i].get("error", "")).lower()]
    if rate_limited:
        print(f"\n[Ttapi Bulk] Retrying {len(rate_limited)} rate-limited prompt(s) one at a time (120s between each)...")
        for idx_idx, idx in enumerate(rate_limited):
            if idx_idx > 0:
                wait = 120 + random.uniform(0, 30)  # 120-150s with jitter
                print(f"  Waiting {wait:.1f}s before retry {idx_idx + 1}/{len(rate_limited)}...")
                time.sleep(wait)
            _, result = process_one_prompt((idx, BULK_PROMPTS[idx]))
            ttapi_results[idx] = result
            urls = result.get("image_urls") or []
            err = result.get("error")
            if err:
                print(f"  Prompt {idx + 1}: still failed - {err}")
            else:
                print(f"  Prompt {idx + 1}: {len(urls)} image(s) (recovered)")
        print("[Ttapi Bulk] Second pass done.")

    # Display all (4 per row per prompt)
    for i, r in enumerate(ttapi_results):
        if r is None:
            continue
        display(HTML(f"<h4>Prompt {i+1}</h4>"))
        urls = r.get("image_urls") or []
        if urls:
            display_images_4_per_row(urls)
        else:
            display(HTML("<p>No images.</p>"))

## 5a. Regenerate failed prompts via ChatGPT (optional)

Run **after** the bulk generation cell (Section 5). If any prompts failed (e.g. **banned prompt words**, 400 Bad Request), this step will:
1. Detect failed prompts (those with an error and no images).
2. Call the **OpenAI Chat Completions API** to rephrase each failed prompt so it keeps the same intent but avoids banned or invalid words.
3. Re-submit the rephrased prompts to TTAPI and update **ttapi_results** in place.

Set **OPENAI_API_KEY** below (get one at [platform.openai.com](https://platform.openai.com/api-keys)). Then run this cell; you can run **5b. Filter** again afterward to include the newly generated images.

In [ ]:
import time
import re
OPENAI_API_KEY = ""  # Get from https://platform.openai.com/api-keys (for ChatGPT rephrasing)
OPENAI_MODEL = "gpt-4o-mini"  # or "gpt-4o" for better rephrasing

def _extract_urls_and_params(prompt_text):
    """Extract image URLs and Midjourney params (--ar, --v, --relax, etc.) from original prompt."""
    urls = re.findall(r'https?://[^\s]+', prompt_text)
    urls = [u.rstrip('.,;)]') for u in urls]
    params = re.findall(r'--\w+(?:\s+[^\s-]+)?', prompt_text)
    return urls, params

def _rephrase_prompt_via_openai(original_prompt, error_message, api_key, model=OPENAI_MODEL):
    """Ask OpenAI to rephrase the text only (no URLs/params). We reattach URLs and params after."""
    import requests as _req
    urls, params = _extract_urls_and_params(original_prompt)
    text_only = original_prompt
    for u in urls:
        text_only = text_only.replace(u, " ").strip()
    for p in params:
        text_only = text_only.replace(p, " ").strip()
    text_only = re.sub(r'\s+', ' ', text_only).strip()
    system = (
        "You are a Midjourney prompt expert. The user will give the TEXT part of a Midjourney prompt (no image URLs, no --parameters) and an error. "
        "Reply with a single rephrased description that keeps the same visual intent but avoids banned/blocked words from the error. "
        "Output ONLY the new text description, no URLs, no --parameters, no explanation, no quotes."
    )
    user = f"Text to rephrase: {text_only}\n\nError from API: {error_message}"
    r = _req.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}], "max_tokens": 300},
        timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    content = (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""
    new_text = content.strip().strip('"').strip("'").strip()
    if not new_text:
        return None
    parts = urls + [new_text] + params
    return " ".join(p for p in parts if p)

try:
    _ = ttapi_results
    _ = BULK_PROMPTS
except NameError:
    ttapi_results = []
    BULK_PROMPTS = []

failed_indices = [i for i, r in enumerate(ttapi_results) if r and r.get("error") and not (r.get("image_urls"))]

if not failed_indices:
    print("No failed prompts to regenerate. Run Section 5 (Bulk) first; if all prompts succeeded, this step is skipped.")
elif not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY above to rephrase failed prompts via ChatGPT.")
    print(f"Failed prompt indices (1-based): {[i+1 for i in failed_indices]}")
else:
    n_failed = len(failed_indices)
    print(f"Found {n_failed} failed prompt(s). Rephrasing via OpenAI ({OPENAI_MODEL}) then re-submitting to TTAPI...")
    new_prompts = {}
    for i in failed_indices:
        orig = BULK_PROMPTS[i] if i < len(BULK_PROMPTS) else (ttapi_results[i].get("prompt") or "")
        err = ttapi_results[i].get("error") or "Unknown error"
        try:
            new_p = _rephrase_prompt_via_openai(orig, err, OPENAI_API_KEY, OPENAI_MODEL)
            if new_p:
                new_prompts[i] = new_p
                preview = new_p[:70] + "..." if len(new_p) > 70 else new_p
                print(f"  Prompt {i+1}: rephrased -> {preview}")
            else:
                print(f"  Prompt {i+1}: OpenAI returned empty; keeping failed.")
        except Exception as e:
            print(f"  Prompt {i+1}: OpenAI error - {e}")

    if new_prompts:
        print(f"\nRe-submitting {len(new_prompts)} rephrased prompt(s) to TTAPI...")
        for idx, new_prompt in new_prompts.items():
            _, result = process_one_prompt((idx, new_prompt))
            ttapi_results[idx] = result
            time.sleep(1)  # gentle delay between re-submits
            urls = result.get("image_urls") or []
            err = result.get("error")
            if err:
                print(f"  Prompt {idx + 1}: still failed - {err}")
            else:
                print(f"  Prompt {idx + 1}: {len(urls)} image(s)")
        print("Done. Run 5b (Filter) again to include newly generated images.")
    else:
        print("No prompts were rephrased. Check OPENAI_API_KEY and errors above.")

## 5b. Filter by position (1st / 2nd / 3rd / 4th / All)

Same as the app’s **Your Collection**: choose which image(s) from each prompt to use. Each prompt returns 1–4 images; select positions 1–4 (1st, 2nd, 3rd, 4th). **filtered_image_urls** is used for Download, Drive upload, and Grid Generator (shuffled).

In [ ]:
import random

try:
    _ = ttapi_results
except NameError:
    ttapi_results = []

# Which position(s) to keep from each prompt: 1=1st, 2=2nd, 3=3rd, 4=4th. Default [1,2,3,4] = All.
SELECTED_POSITIONS = [1, 2, 3, 4]  # e.g. [2] = only 2nd image per prompt; [1, 3] = 1st and 3rd

filtered_image_urls = []
for r in ttapi_results:
    urls = r.get("image_urls") or []
    for pos in SELECTED_POSITIONS:
        if 1 <= pos <= len(urls):
            filtered_image_urls.append(urls[pos - 1])

random.shuffle(filtered_image_urls)  # Same as app: shuffle so images aren't grouped by prompt
print(f"Filtered: {len(filtered_image_urls)} image(s) (positions {sorted(SELECTED_POSITIONS)} from each of {len(ttapi_results)} prompts)")

## 6. Download generated images (optional)

Download **filtered** images (from step 5b) as a ZIP. Uses shuffled order like the app.

In [ ]:
import zipfile
import io

try:
    _ = filtered_image_urls
except NameError:
    filtered_image_urls = []

if not filtered_image_urls:
    print("Run the bulk TTAPI cell and the filter cell (5b) first.")
else:
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for i, url in enumerate(filtered_image_urls):
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                zf.writestr(f"image_{i+1:03d}.png", resp.content)
            except Exception as e:
                print(f"Skip {url}: {e}")
    zip_path = "/tmp/midjourney_ttapi_images.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getvalue())
    from google.colab import files
    files.download(zip_path)
    print("Downloaded midjourney_ttapi_images.zip")

## 7. Upload to Google Drive (optional)

Upload **filtered** images (from step 5b) to a **new** Drive folder: **random order**, **random filenames**, **JPG** format. You will be asked for the folder name; it is created under the parent folder.

**Auth options:**
- **Colab auth (recommended):** Set `USE_COLAB_DRIVE_AUTH = True`. Run the cell; a browser sign-in will open. No Client ID/Secret needed.
- **Instant upload (Colab + Drive mount):** With Colab auth, set `DRIVE_USE_MOUNT = True` and **mount Drive first** in a cell: `from google.colab import drive; drive.mount('/content/drive')`. Then upload writes directly to your Drive (no per-file HTTP) — much faster.
- **API upload:** If Drive is not mounted (or `DRIVE_USE_MOUNT = False`), files are uploaded via the Drive API (parallel workers). Use `GOOGLE_DRIVE_PARENT_FOLDER_ID` to choose a parent folder.
- **OAuth:** Set `USE_COLAB_DRIVE_AUTH = False` and fill Client ID, Secret, Refresh Token, and parent folder ID (same as Arcane Splitter).

In [ ]:
USE_COLAB_DRIVE_AUTH = True   # True = sign in with Google in Colab (no Client ID/Secret). False = use OAuth below.
DRIVE_USE_MOUNT = True   # When Colab auth: if True and Drive is mounted, write to mount (instant). Run: from google.colab import drive; drive.mount('/content/drive')
DRIVE_MOUNT_SUBFOLDER = ""   # Optional subfolder under My Drive, e.g. "Colab" or "Colab/Midjourney". Only used when DRIVE_USE_MOUNT.
GOOGLE_DRIVE_CLIENT_ID = ""
GOOGLE_DRIVE_CLIENT_SECRET = ""
GOOGLE_DRIVE_REFRESH_TOKEN = ""
GOOGLE_DRIVE_PARENT_FOLDER_ID = ""   # Parent folder ID (from Drive URL). Used for API uploads; when using mount we resolve it to path.

def _drive_refresh_token(client_id, client_secret, refresh_token):
    r = requests.post(
        "https://oauth2.googleapis.com/token",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"client_id": client_id, "client_secret": client_secret,
              "refresh_token": refresh_token, "grant_type": "refresh_token"},
        timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]

def _drive_get_access_token():
    """Get Drive access token: Colab auth (browser sign-in) or OAuth refresh token."""
    if USE_COLAB_DRIVE_AUTH:
        from google.colab import auth
        auth.authenticate_user()
        import google.auth
        from google.auth.transport.requests import Request
        credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
        credentials.refresh(Request())
        return credentials.token
    return _drive_refresh_token(GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN)

def _drive_folder_id_to_mount_path(access_token, folder_id):
    """Resolve a Drive folder ID to its path under My Drive (for mount). Returns path like 'Parent/Sub' or '' for root. None if not found."""
    if not folder_id or (folder_id or "").strip() in ("", "root"):
        return ""
    folder_id = folder_id.strip()
    names = []
    fid = folder_id
    while fid and fid != "root":
        try:
            r = requests.get(
                f"https://www.googleapis.com/drive/v3/files/{fid}",
                headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
                params={"fields": "name,parents"},
                timeout=15)
            r.raise_for_status()
            data = r.json()
            names.insert(0, (data.get("name") or "").strip() or "Folder")
            parents = data.get("parents") or []
            fid = parents[0] if parents else None
        except Exception:
            return None
    return "/".join(n for n in names if n)

def _drive_create_folder(access_token, parent_folder_id, folder_name):
    """Create a new folder in Drive under parent. Returns the new folder id."""
    r = requests.post(
        "https://www.googleapis.com/drive/v3/files",
        headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
        json={"name": folder_name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_folder_id]},
        timeout=30)
    r.raise_for_status()
    return r.json()["id"]

def _drive_upload_file(access_token, folder_id, filename, file_bytes, mime="image/png"):
    """Upload one file to Drive. Same as arcane_splitter.google_drive_upload_file: returns id, name, webViewLink."""
    import json
    import secrets
    boundary = "-------" + secrets.token_hex(8)
    metadata = {"name": filename, "parents": [folder_id]}
    body = (
        f"--{boundary}\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n"
        + json.dumps(metadata)
        + f"\r\n--{boundary}\r\nContent-Type: {mime}\r\n\r\n"
    ).encode("utf-8") + file_bytes + f"\r\n--{boundary}--".encode("utf-8")
    r = requests.post(
        "https://www.googleapis.com/upload/drive/v3/files?uploadType=multipart",
        headers={"Authorization": f"Bearer {access_token}",
                 "Content-Type": f"multipart/related; boundary={boundary}"},
        data=body, timeout=120)
    r.raise_for_status()
    data = r.json()
    return {
        "id": data["id"],
        "name": data.get("name", filename),
        "webViewLink": f"https://drive.google.com/file/d/{data['id']}/view",
    }

import random
import secrets
import io
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_UPLOAD_WORKERS = 10   # Upload this many images in parallel (faster)

def _download_convert_upload_one(args):
    """Download image from URL, convert to JPG, upload to Drive. Returns (result_dict, None) or (None, error)."""
    url, token, folder_id = args
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=92)
        jpg_bytes = buf.getvalue()
        filename = f"{secrets.token_hex(8)}.jpg"
        d = _drive_upload_file(token, folder_id, filename, jpg_bytes, mime="image/jpeg")
        return (d, None)
    except Exception as e:
        return (None, str(e))

try:
    _ = filtered_image_urls
except NameError:
    filtered_image_urls = []

_drive_ready = USE_COLAB_DRIVE_AUTH or all([GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN, GOOGLE_DRIVE_PARENT_FOLDER_ID])
if not filtered_image_urls or not _drive_ready:
    print("Run bulk TTAPI + filter (5b) first. If using OAuth (USE_COLAB_DRIVE_AUTH=False), set Client ID, Secret, Refresh Token, and parent folder ID.")
else:
    folder_name = input("Name for the new folder where images will be uploaded: ").strip()
    if not folder_name:
        print("No folder name entered. Skipping upload.")
    else:
        to_upload = list(filtered_image_urls)
        random.shuffle(to_upload)
        n = len(to_upload)
        import os
        use_mount = USE_COLAB_DRIVE_AUTH and DRIVE_USE_MOUNT and os.path.exists("/content/drive/MyDrive")
        if use_mount:
            base = "/content/drive/MyDrive"
            parent_id = (GOOGLE_DRIVE_PARENT_FOLDER_ID or "").strip()
            if parent_id and parent_id != "root":
                token = _drive_get_access_token()
                parent_path = _drive_folder_id_to_mount_path(token, parent_id)
                if parent_path:
                    base = os.path.join(base, parent_path)
            elif (DRIVE_MOUNT_SUBFOLDER or "").strip():
                base = os.path.join(base, (DRIVE_MOUNT_SUBFOLDER or "").strip().strip("/"))
            safe_name = folder_name.replace("/", "_").replace("\\", "_").strip() or "Midjourney"
            folder_path = os.path.join(base, safe_name)
            os.makedirs(folder_path, exist_ok=True)
            print(f'Using mounted Drive (instant). Writing {n} image(s) to "{folder_path}"...')
            ok = 0
            for url in to_upload:
                try:
                    resp = requests.get(url, timeout=30)
                    resp.raise_for_status()
                    img = Image.open(io.BytesIO(resp.content)).convert("RGB")
                    buf = io.BytesIO()
                    img.save(buf, format="JPEG", quality=92)
                    filename = f"{secrets.token_hex(8)}.jpg"
                    with open(os.path.join(folder_path, filename), "wb") as f:
                        f.write(buf.getvalue())
                    ok += 1
                except Exception as e:
                    print(f"Failed {url}: {e}")
            print(f"Done. {ok}/{n} written. Open Drive: {folder_path}")
        else:
            token = _drive_get_access_token()
            parent_id = (GOOGLE_DRIVE_PARENT_FOLDER_ID or "").strip() or "root"
            folder_id = _drive_create_folder(token, parent_id, folder_name)
            print(f'Created folder "{folder_name}". Uploading {n} image(s) as JPG in parallel ({min(DRIVE_UPLOAD_WORKERS, n)} workers)...')
            drive_links = []
            args_list = [(url, token, folder_id) for url in to_upload]
            with ThreadPoolExecutor(max_workers=min(DRIVE_UPLOAD_WORKERS, n)) as ex:
                futures = {ex.submit(_download_convert_upload_one, a): a for a in args_list}
                for future in as_completed(futures):
                    d, err = future.result()
                    if d:
                        drive_links.append(d)
                    else:
                        print(f"Upload failed: {err}")
            print(f"Done. {len(drive_links)}/{n} uploaded. Folder: https://drive.google.com/drive/folders/{folder_id}")

## 8. Grid Generator (optional)

Same as the app: create **3 rows × 4 columns = 12 images per grid**. Uses **filtered** images (from step 5b), shuffled. Set **NUM_GRID_PAGES** (max = filtered count ÷ 12). Output: grid images displayed and downloaded as a ZIP.

In [ ]:
from PIL import Image, ImageOps
import io
import zipfile

try:
    _ = filtered_image_urls
except NameError:
    filtered_image_urls = []

NUM_GRID_PAGES = 1  # Max: len(filtered_image_urls) // 12
GRID_SIZE = (3000, 3000)  # Same as app
ROWS, COLS = 3, 4
PADDING = 20
IMAGES_PER_GRID = ROWS * COLS

if len(filtered_image_urls) < IMAGES_PER_GRID:
    print(f"Need at least {IMAGES_PER_GRID} filtered images. You have {len(filtered_image_urls)}. Run filter (5b) with more positions or more prompts.")
else:
    urls_for_grid = list(filtered_image_urls)
    random.shuffle(urls_for_grid)
    max_pages = len(urls_for_grid) // IMAGES_PER_GRID
    num_pages = min(NUM_GRID_PAGES, max_pages)

    cell_w = (GRID_SIZE[0] - PADDING * (COLS + 1)) // COLS
    cell_h = (GRID_SIZE[1] - PADDING * (ROWS + 1)) // ROWS
    grid_images = []

    for page in range(num_pages):
        base = Image.new("RGB", GRID_SIZE, (255, 255, 255))
        for row in range(ROWS):
            for col in range(COLS):
                idx = page * IMAGES_PER_GRID + row * COLS + col
                if idx >= len(urls_for_grid):
                    break
                resp = requests.get(urls_for_grid[idx], timeout=30)
                resp.raise_for_status()
                img = Image.open(io.BytesIO(resp.content)).convert("RGB")
                # Cover-fit: fill cell, center crop (same as app)
                img = ImageOps.fit(img, (cell_w, cell_h), Image.Resampling.LANCZOS)
                x = PADDING + col * (cell_w + PADDING)
                y = PADDING + row * (cell_h + PADDING)
                base.paste(img, (x, y))
        grid_images.append(base)

    # Display
    from IPython.display import display, HTML
    for i, g in enumerate(grid_images):
        display(HTML(f"<h4>Grid {i+1}</h4>"))
        display(g)

    # Download as ZIP
    zip_buf = io.BytesIO()
    with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
        for i, g in enumerate(grid_images):
            buf = io.BytesIO()
            g.save(buf, format="PNG")
            zf.writestr(f"grid_{i+1}.png", buf.getvalue())
    zip_buf.seek(0)
    grid_zip_path = "/tmp/midjourney_grids.zip"
    with open(grid_zip_path, "wb") as f:
        f.write(zip_buf.getvalue())
    from google.colab import files
    files.download(grid_zip_path)
    print(f"Downloaded {len(grid_images)} grid(s) as midjourney_grids.zip")

## 8b. Upload grids to Google Drive (random order, random names, JPG)

Run **Section 8** first to create **grid_images**. Uses the same Drive auth as Section 7 (Colab sign-in or OAuth). You will be asked for the **name of the new folder**; grids are uploaded there (under the parent folder from Section 7, or root) in **random order** with **random filenames** as **JPG**.

In [ ]:
import secrets

try:
    _ = grid_images
except NameError:
    grid_images = []

try:
    _drive_ready_8b = USE_COLAB_DRIVE_AUTH or all([GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN, GOOGLE_DRIVE_PARENT_FOLDER_ID])
except NameError:
    _drive_ready_8b = all([GOOGLE_DRIVE_CLIENT_ID, GOOGLE_DRIVE_CLIENT_SECRET, GOOGLE_DRIVE_REFRESH_TOKEN, GOOGLE_DRIVE_PARENT_FOLDER_ID])
if not grid_images or not _drive_ready_8b:
    print("Run Section 8 (Grid Generator) first. Run Section 7 to set USE_COLAB_DRIVE_AUTH / Drive credentials.")
else:
    folder_name = input("Name for the new folder where grids will be uploaded: ").strip()
    if not folder_name:
        print("No folder name entered. Skipping upload.")
    else:
        to_upload = list(grid_images)
        random.shuffle(to_upload)
        n = len(to_upload)
        import os
        use_mount = USE_COLAB_DRIVE_AUTH and DRIVE_USE_MOUNT and os.path.exists("/content/drive/MyDrive")
        if use_mount:
            base = "/content/drive/MyDrive"
            parent_id = (GOOGLE_DRIVE_PARENT_FOLDER_ID or "").strip()
            if parent_id and parent_id != "root":
                token = _drive_get_access_token()
                parent_path = _drive_folder_id_to_mount_path(token, parent_id)
                if parent_path:
                    base = os.path.join(base, parent_path)
            elif (DRIVE_MOUNT_SUBFOLDER or "").strip():
                base = os.path.join(base, (DRIVE_MOUNT_SUBFOLDER or "").strip().strip("/"))
            safe_name = folder_name.replace("/", "_").replace("\\", "_").strip() or "Grids"
            folder_path = os.path.join(base, safe_name)
            os.makedirs(folder_path, exist_ok=True)
            print(f'Using mounted Drive (instant). Writing {n} grid(s) to "{folder_path}"...')
            ok = 0
            for img in to_upload:
                try:
                    buf = io.BytesIO()
                    img.save(buf, format="JPEG", quality=92)
                    filename = f"{secrets.token_hex(8)}.jpg"
                    with open(os.path.join(folder_path, filename), "wb") as f:
                        f.write(buf.getvalue())
                    ok += 1
                except Exception as e:
                    print(f"Failed: {e}")
            print(f"Done. {ok}/{n} written. Open Drive: {folder_path}")
        else:
            token = _drive_get_access_token()
            parent_id = (GOOGLE_DRIVE_PARENT_FOLDER_ID or "").strip() or "root"
            folder_id = _drive_create_folder(token, parent_id, folder_name)
            try:
                workers = min(DRIVE_UPLOAD_WORKERS, n)
            except NameError:
                workers = min(10, n)
            print(f'Created folder "{folder_name}". Uploading {n} grid(s) as JPG in parallel ({workers} workers)...')
            drive_links = []
            def _upload_one_grid(args):
                img, tok, fid = args
                try:
                    buf = io.BytesIO()
                    img.save(buf, format="JPEG", quality=92)
                    jpg_bytes = buf.getvalue()
                    filename = f"{secrets.token_hex(8)}.jpg"
                    return (_drive_upload_file(tok, fid, filename, jpg_bytes, mime="image/jpeg"), None)
                except Exception as e:
                    return (None, str(e))
            args_list = [(img, token, folder_id) for img in to_upload]
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futures = {ex.submit(_upload_one_grid, a): a for a in args_list}
                for future in as_completed(futures):
                    d, err = future.result()
                    if d:
                        drive_links.append(d)
                    else:
                        print(f"Upload failed: {err}")
            print(f"Done. {len(drive_links)}/{n} uploaded. Folder: https://drive.google.com/drive/folders/{folder_id}")